# OptiVision easy mode: a folder of PDFs to a compressed, searchable index

This notebook builds an index of the bundled 20-page sample PDF, searches it and reads the report. It runs on a Colab T4 (Runtime, Change runtime type, T4 GPU). To use your own PDFs, put them in a folder and change `FOLDER`.

Every number the package prints comes from our own validation sets and has not been checked on new documents. The package prints that sentence under every plan and every build. The full guide is [docs/package/guide/EASY.md](https://github.com/Daemon-VI/optivision-rag/blob/main/docs/package/guide/EASY.md).

## 1. Install

The `easy-st` extra brings the loader of the example model. If the install changes a library that is already imported, restart the runtime (Runtime, Restart session) and run the cells again from the top.

In [ ]:
%pip install -q "optivision-rag[easy-st]"

## 2. Check the machine

`doctor` reports Python, the torch CUDA build, the GPU, free disk and known problems of your model. Fix every line marked `Problem` before you build.

In [ ]:
import optivision.easy as ov

MODEL = "vidore/colqwen2-v1.0-hf"   # a registry id: ov.options() lists every model
TARGET = 0.97                       # the share of full-precision nDCG@5 to keep; not "97% of answers right"

report = ov.doctor(MODEL)
print(report)

## 3. Choose where the temporary files go

The build writes a temporary float16 copy of the pages. On Google Drive a build that is interrupted resumes in a new session. Mount Drive and keep `WORK_DIR`, or set `WORK_DIR = None` to work in the local session only.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
WORK_DIR = "/content/drive/MyDrive/optivision_work"   # None keeps the temporary files in this session

## 4. The folder

The next cell copies the bundled sample PDF (20 pages of public-domain text, written for the package) into `my_pdfs`. Delete it and put your own PDFs there to index them.

In [ ]:
import pathlib
import shutil

FOLDER = "my_pdfs"
pathlib.Path(FOLDER).mkdir(exist_ok=True)
shutil.copy(pathlib.Path(ov.__file__).parent / "data" / "sample.pdf", pathlib.Path(FOLDER) / "sample.pdf")

## 5. Plan before you build

`options` needs no GPU and downloads nothing. It names the layout the rule picks for this model and target, how big the index will be and how long encoding takes. Read the `lowest kept` line: it is a catalogue number from our benchmarks, not a promise for your documents.

In [ ]:
plan = ov.options(MODEL, target=TARGET, pages=FOLDER)
print(plan)

## 6. Build

`build` encodes every page, picks the layout, saves `my_pdfs.ovx` next to the folder and prints a summary. The same call run again adds only new and changed files.

In [ ]:
index = ov.build(FOLDER, model=MODEL, target=TARGET, work_dir=WORK_DIR)

## 7. Search

Each hit has the page id, the file, the 1-based page number, a score and the page text. `hit.image()` re-renders the page, which is what you pass to a vision-language model.

In [ ]:
hits = index.search("How do I get a golden yellow dye on wool?", k=3)
for hit in hits:
    print(hit.page, hit.id, round(hit.score, 2))

page = hits[0].image()
page.thumbnail((600, 800))
page

## 8. Hand the page to a vision-language model

This step depends on the model you use, so the notebook does not run one. With any model that takes an image and a question, pass it `hits[0].image()` and the question. For example:

```python
# page = hits[0].image()
# answer = your_vlm(images=[page], text="How do I get a golden yellow dye on wool?")
```

## 9. Read the report

`report()` lists every number with the tag that says where it comes from. `save` writes `.html`, `.md` or `.json`.

In [ ]:
report = index.report()
report.save("my_report.md")
print([title for title, block in report.sections])

## 10. Check it later

`check()` is a spot check: it compares the build record with the model now loaded, looks for drift and asks whether the layout still qualifies at today's page count. It changes nothing. It does not retest search quality on your documents.

In [ ]:
check = index.check()
print(check)

## Next

- Add documents: `index.add("more_pdfs/")`.
- Open the index in a new session: `index = ov.load("my_pdfs.ovx")`.
- Get a smaller index than the catalogue allows, with your own real questions: `index.upgrade(questions)` (build with `keep_float16=True` first). See the guide for what it needs.
- Read the guide's Limitations section before you rely on a number: it lists the validation tests that did not pass.